Bronze: Import dataset from Kaggle

In [0]:
import pandas as pd
import kagglehub as kh
import os

# Download latest version
path = kh.dataset_download("binayas/f1-dataset")

print("Path to dataset files:", path)

dataset_path = "/home/spark-6c0e28bd-0f58-426e-9d18-ab/.cache/kagglehub/datasets/binayas/f1-dataset/versions/12"

# List all files in the dataset (recursively)
for root, dirs, files in os.walk(dataset_path):
    for f in sorted(files):
        rel = os.path.relpath(os.path.join(root, f), dataset_path)
        size_mb = os.path.getsize(os.path.join(root, f)) / 1024 / 1024
        print(f"{rel}  ({size_mb:.2f} MB)")

Path to dataset files: /home/spark-49a18e8a-2705-4264-9cd1-74/.cache/kagglehub/datasets/binayas/f1-dataset/versions/12


Bronze: Load all dataset tables

In [0]:
import glob
import os

from pyspark.sql import SparkSession

# Read each small CSV with pandas on the driver, then hand it to Spark once.
dataset_path = path

csv_files = sorted(glob.glob(os.path.join(dataset_path, "*.csv")))
table_names = [os.path.splitext(os.path.basename(f))[0] for f in csv_files]
print("Tables found:", table_names)

# Ergast CSVs use \N for missing values - read them as proper nulls.
dfs = {}
for f, name in zip(csv_files, table_names):
    pdf = pd.read_csv(f, na_values="\\N")
    dfs[name] = spark.createDataFrame(pdf)
    print(f"{name}: {dfs[name].count()} rows, {len(dfs[name].columns)} columns")

print("\nCurrent catalog:", spark.catalog.currentCatalog(), "| schema:", spark.catalog.currentDatabase())

Tables found: ['circuits', 'constructor_standings', 'constructors', 'driver_standings', 'drivers', 'lap_times', 'pit_stops', 'qualifying', 'races', 'results', 'seasons', 'sprint_results', 'status']
circuits: 78 rows, 10 columns
constructor_standings: 13807 rows, 8 columns
constructors: 214 rows, 6 columns
driver_standings: 35718 rows, 8 columns
drivers: 881 rows, 10 columns
lap_times: 634079 rows, 7 columns
pit_stops: 12856 rows, 8 columns
qualifying: 11320 rows, 10 columns
races: 1172 rows, 19 columns
results: 27590 rows, 19 columns
seasons: 77 rows, 3 columns
sprint_results: 590 rows, 17 columns
status: 141 rows, 3 columns

Current catalog: workspace | schema: default


Bronze: Save tables into f1_bronze schema

In [0]:
from pyspark.sql.functions import current_timestamp

# Create the f1_bronze schema
spark.sql("CREATE SCHEMA IF NOT EXISTS f1_bronze")

# Save every table into the f1_bronze schema
# Add _ingested_at timestamp for load provenance and enable Delta CDF so future incremental runs can detect what changed (inserts/updates/deletes).
# Write mode is APPEND - each run adds the latest Kaggle snapshot as a new batch.
# Duplicates are expected (same rows re-loaded); silver layer deduplicates via MERGE.
for name, df in dfs.items():
    df.withColumn("_ingested_at", current_timestamp()) \
     .write.format("delta") \
     .option("mergeSchema", "true") \
     .mode("append") \
     .saveAsTable(f"f1_bronze.{name}")
    print(f"Appended to f1_bronze.{name} (with _ingested_at)")

# Enable Delta Change Data Feed (CDF) on all bronze tables via ALTER TABLE
for name in dfs.keys():
    spark.sql(f"ALTER TABLE f1_bronze.{name} SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
    print(f"CDF enabled on f1_bronze.{name}")

# Verify
display(spark.sql("SHOW TABLES IN f1_bronze"))

# Quick check: show CDF + ingestion column on one table
display(spark.sql("DESCRIBE TABLE EXTENDED f1_bronze.results").filter("col_name IN ('_ingested_at', 'delta.enableChangeDataFeed')"))

Appended to f1_bronze.circuits (with _ingested_at)
Appended to f1_bronze.constructor_standings (with _ingested_at)
Appended to f1_bronze.constructors (with _ingested_at)
Appended to f1_bronze.driver_standings (with _ingested_at)
Appended to f1_bronze.drivers (with _ingested_at)
Appended to f1_bronze.lap_times (with _ingested_at)
Appended to f1_bronze.pit_stops (with _ingested_at)
Appended to f1_bronze.qualifying (with _ingested_at)
Appended to f1_bronze.races (with _ingested_at)
Appended to f1_bronze.results (with _ingested_at)
Appended to f1_bronze.seasons (with _ingested_at)
Appended to f1_bronze.sprint_results (with _ingested_at)
Appended to f1_bronze.status (with _ingested_at)
CDF enabled on f1_bronze.circuits
CDF enabled on f1_bronze.constructor_standings
CDF enabled on f1_bronze.constructors
CDF enabled on f1_bronze.driver_standings
CDF enabled on f1_bronze.drivers
CDF enabled on f1_bronze.lap_times
CDF enabled on f1_bronze.pit_stops
CDF enabled on f1_bronze.qualifying
CDF enable

database,tableName,isTemporary
f1_bronze,circuits,false
f1_bronze,constructor_standings,false
f1_bronze,constructors,false
f1_bronze,driver_standings,false
f1_bronze,drivers,false
f1_bronze,lap_times,false
f1_bronze,pit_stops,false
f1_bronze,qualifying,false
f1_bronze,races,false
f1_bronze,results,false


col_name,data_type,comment
_ingested_at,timestamp,null


Silver: CDF merge helper

In [0]:
from pyspark.sql.functions import col, row_number, desc
from pyspark.sql.window import Window

# Create silver schema
spark.sql("CREATE SCHEMA IF NOT EXISTS f1_silver")

# Metadata table to track the last CDF version processed per bronze table
spark.sql("""
    CREATE TABLE IF NOT EXISTS f1_silver._cdf_metadata (
        table_name STRING,
        last_version BIGINT,
        last_processed_at TIMESTAMP
    )
""")

def merge_from_cdf(bronze_table, silver_table, key_cols, transform_fn=None):
    """
    Incrementally sync a bronze table into silver using Delta CDF.

    First run: reads the full bronze table, deduplicates by natural key
    (keeping latest _ingested_at), and overwrites silver.
    Subsequent runs: reads only CDF changes since the last run, deduplicates
    by natural key (keeping latest _commit_version), and MERGEs into silver.

    Args:
        bronze_table:  fully qualified bronze table name (e.g. "f1_bronze.circuits")
        silver_table:  fully qualified silver table name (e.g. "f1_silver.circuits")
        key_cols:      list of natural key column names (e.g. ["circuitId"])
        transform_fn:  optional function(df) -> df for table-specific cleaning
    """
    # Get the last processed version for this table
    row = spark.sql(
        f"SELECT COALESCE(MAX(last_version), 0) FROM f1_silver._cdf_metadata WHERE table_name = '{bronze_table}'"
    ).collect()[0]
    last_version = row[0] if row[0] is not None else 0

    if last_version == 0:
        # ── First run: full load from bronze (CDF not available from version 0) ──
        source = spark.table(bronze_table)
        # Deduplicate by natural key, keeping latest _ingested_at
        w = Window.partitionBy(*key_cols).orderBy(desc("_ingested_at"))
        deduped = (
            source.withColumn("_rn", row_number().over(w))
            .filter("_rn = 1")
            .drop("_rn")
        )
        # Apply table-specific transformations (drops _ingested_at + does cleaning)
        if transform_fn:
            deduped = transform_fn(deduped)
        cnt = deduped.count()
        # Overwrite silver with clean deduplicated data
        deduped.write.format("delta") \
            .option("overwriteSchema", "true") \
            .mode("overwrite") \
            .saveAsTable(silver_table)
        # Record the latest Delta table version as the CDF starting point
        max_version = spark.sql(f"DESCRIBE HISTORY {bronze_table}").selectExpr("MAX(version) AS v").collect()[0]["v"]
        action = "Overwrote"
        starting = 0
    else:
        # ── Subsequent runs: incremental CDF read + MERGE ──
        starting = last_version + 1
        changes = (
            spark.read.format("delta")
            .option("readChangeFeed", "true")
            .option("startingVersion", starting)
            .table(bronze_table)
            .filter("_change_type IN ('insert', 'update_postimage')")
        )
        cnt = changes.count()
        if cnt == 0:
            print(f"  No changes for {silver_table} since CDF version {last_version}")
            return
        # Deduplicate by natural key, keeping latest commit version
        w = Window.partitionBy(*key_cols).orderBy(desc("_commit_version"))
        deduped = (
            changes.withColumn("_rn", row_number().over(w))
            .filter("_rn = 1")
            .drop("_rn")
            .drop("_change_type", "_commit_version", "_commit_timestamp")
        )
        if transform_fn:
            deduped = transform_fn(deduped)
        # MERGE into silver
        temp_view = silver_table.replace(".", "_")
        deduped.createOrReplaceTempView(temp_view)
        on_clause = " AND ".join([f"t.{c} = s.{c}" for c in key_cols])
        spark.sql(f"""
            MERGE INTO {silver_table} AS t
            USING {temp_view} AS s
            ON {on_clause}
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
        max_version = changes.selectExpr("MAX(_commit_version) AS v").collect()[0]["v"]
        action = "Merged"

    # Update metadata with the latest version processed
    spark.sql(f"""
        MERGE INTO f1_silver._cdf_metadata AS t
        USING (SELECT '{bronze_table}' AS table_name, {max_version} AS last_version, current_timestamp() AS last_processed_at) AS s
        ON t.table_name = s.table_name
        WHEN MATCHED   THEN UPDATE SET last_version = s.last_version, last_processed_at = s.last_processed_at
        WHEN NOT MATCHED THEN INSERT *
    """)

    print(f"  {action} {cnt} rows into {silver_table}  (Delta v{starting} -> v{max_version})")

Silver: Clean circuits

In [0]:
from pyspark.sql.functions import col, when

# Create silver schema (if not already created by helper cell)
spark.sql("CREATE SCHEMA IF NOT EXISTS f1_silver")

# Clean circuits: drop url/source, fill known null altitude for Madring (Madrid ~667m)
# merge_from_cdf: 1st run = full overwrite (creates table), subsequent runs = incremental MERGE
merge_from_cdf(
    "f1_bronze.circuits",
    "f1_silver.circuits",
    ["circuitId"],
    lambda df: df.drop("url", "source", "_ingested_at")
        .withColumn("alt", when(col("circuitId") == 81, 667.0).otherwise(col("alt")))
)
print("f1_silver.circuits ready")
display(spark.table("f1_silver.circuits").limit(10))

  Merged 156 rows into f1_silver.circuits  (Delta v10 -> v12)
f1_silver.circuits ready


circuitId,circuitRef,name,location,country,lat,lng,alt
1,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.8497,144.968,10.0
2,sepang,Sepang International Circuit,Kuala Lumpur,Malaysia,2.76083,101.738,18.0
3,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,7.0
4,catalunya,Circuit de Barcelona-Catalunya,Montmeló,Spain,41.57,2.26111,109.0
5,istanbul,Istanbul Park,Istanbul,Turkey,40.9517,29.405,130.0
6,monaco,Circuit de Monaco,Monte-Carlo,Monaco,43.7347,7.42056,7.0
7,villeneuve,Circuit Gilles Villeneuve,Montreal,Canada,45.5,-73.5228,13.0
8,magny_cours,Circuit de Nevers Magny-Cours,Magny Cours,France,46.8642,3.16361,228.0
9,silverstone,Silverstone Circuit,Silverstone,UK,52.0786,-1.01694,153.0
10,hockenheimring,Hockenheimring,Hockenheim,Germany,49.3278,8.56583,103.0


Silver: Clean constructor_standings

In [0]:
# Drop source (provenance) and positionText (redundant string copy of position)
# merge_from_cdf: 1st run = full overwrite (creates table), subsequent runs = incremental MERGE
merge_from_cdf(
    "f1_bronze.constructor_standings",
    "f1_silver.constructor_standings",
    ["constructorStandingsId"],
    lambda df: df.drop("source", "positionText", "_ingested_at")
)
print("f1_silver.constructor_standings ready")
display(spark.table("f1_silver.constructor_standings").limit(10))

  Merged 27614 rows into f1_silver.constructor_standings  (Delta v11 -> v13)
f1_silver.constructor_standings ready


constructorStandingsId,raceId,constructorId,points,position,wins
1,18,1,14.0,1,1
2,18,2,8.0,3,0
3,18,3,9.0,2,0
4,18,4,5.0,4,0
5,18,5,2.0,5,0
6,18,6,1.0,6,0
7,19,1,24.0,1,1
8,19,2,19.0,2,0
9,19,3,9.0,4,0
10,19,4,6.0,5,0


Silver: Clean constructors

In [0]:
# Drop url and source columns
# merge_from_cdf: 1st run = full overwrite (creates table), subsequent runs = incremental MERGE
merge_from_cdf(
    "f1_bronze.constructors",
    "f1_silver.constructors",
    ["constructorId"],
    lambda df: df.drop("url", "source", "_ingested_at")
)
print("f1_silver.constructors ready")
display(spark.table("f1_silver.constructors").limit(10))

  Merged 428 rows into f1_silver.constructors  (Delta v10 -> v12)
f1_silver.constructors ready


constructorId,constructorRef,name,nationality
1,mclaren,McLaren,British
2,bmw_sauber,BMW Sauber,German
3,williams,Williams,British
4,renault,Renault,French
5,toro_rosso,Toro Rosso,Italian
6,ferrari,Ferrari,Italian
7,toyota,Toyota,Japanese
8,super_aguri,Super Aguri,Japanese
9,red_bull,Red Bull,Austrian
10,force_india,Force India,Indian


Silver: Clean driver_standings

In [0]:
# Drop source and positionText (redundant string copy of position)
# merge_from_cdf: 1st run = full overwrite (creates table), subsequent runs = incremental MERGE
merge_from_cdf(
    "f1_bronze.driver_standings",
    "f1_silver.driver_standings",
    ["driverStandingsId"],
    lambda df: df.drop("source", "positionText", "_ingested_at")
)
print("f1_silver.driver_standings ready")
display(spark.table("f1_silver.driver_standings").limit(10))

  Merged 71436 rows into f1_silver.driver_standings  (Delta v10 -> v13)
f1_silver.driver_standings ready


driverStandingsId,raceId,driverId,points,position,wins
1,18,1,10.0,1.0,1
2,18,2,8.0,2.0,0
3,18,3,6.0,3.0,0
4,18,4,5.0,4.0,0
5,18,5,4.0,5.0,0
6,18,6,3.0,6.0,0
7,18,7,2.0,7.0,0
8,18,8,1.0,8.0,0
9,19,1,14.0,1.0,1
10,19,2,11.0,3.0,0


Silver: Clean drivers

In [0]:
from pyspark.sql.functions import col, concat, lit, to_date

# Drop url/source, convert dob to DATE, cast number to int, add full_name
# merge_from_cdf: 1st run = full overwrite (creates table), subsequent runs = incremental MERGE
merge_from_cdf(
    "f1_bronze.drivers",
    "f1_silver.drivers",
    ["driverId"],
    lambda df: df.drop("url", "source", "_ingested_at")
        .withColumn("dob", to_date(col("dob"), "yyyy-MM-dd"))
        .withColumn("number", col("number").cast("int"))
        .withColumn("full_name", concat(col("forename"), lit(" "), col("surname")))
)
print("f1_silver.drivers ready")
display(spark.table("f1_silver.drivers").limit(10))

  Merged 1762 rows into f1_silver.drivers  (Delta v10 -> v12)
f1_silver.drivers ready


driverId,driverRef,number,code,forename,surname,dob,nationality,full_name
1,hamilton,44,HAM,Lewis,Hamilton,1985-01-07,British,Lewis Hamilton
2,heidfeld,null,HEI,Nick,Heidfeld,1977-05-10,German,Nick Heidfeld
3,rosberg,6,ROS,Nico,Rosberg,1985-06-27,German,Nico Rosberg
4,alonso,14,ALO,Fernando,Alonso,1981-07-29,Spanish,Fernando Alonso
5,kovalainen,null,KOV,Heikki,Kovalainen,1981-10-19,Finnish,Heikki Kovalainen
6,nakajima,null,NAK,Kazuki,Nakajima,1985-01-11,Japanese,Kazuki Nakajima
7,bourdais,null,BOU,Sébastien,Bourdais,1979-02-28,French,Sébastien Bourdais
8,raikkonen,7,RAI,Kimi,Räikkönen,1979-10-17,Finnish,Kimi Räikkönen
9,kubica,88,KUB,Robert,Kubica,1984-12-07,Polish,Robert Kubica
10,glock,null,GLO,Timo,Glock,1982-03-18,German,Timo Glock


Silver: Clean lap_times

In [0]:
# Drop source and time string (milliseconds column already has the value)
# merge_from_cdf: 1st run = full overwrite (creates table), subsequent runs = incremental MERGE
merge_from_cdf(
    "f1_bronze.lap_times",
    "f1_silver.lap_times",
    ["raceId", "driverId", "lap"],
    lambda df: df.drop("source", "time", "_ingested_at")
)
print("f1_silver.lap_times ready")
display(spark.table("f1_silver.lap_times").limit(10))

  Merged 1268158 rows into f1_silver.lap_times  (Delta v11 -> v13)
f1_silver.lap_times ready


raceId,driverId,lap,position,milliseconds
1,1,26,10,92198
1,2,45,15,94324
1,3,50,6,90277
1,7,33,15,91551
1,8,41,8,89996
1,10,10,13,91589
1,10,47,6,89983
1,12,11,7,91748
1,17,48,16,92184
1,21,42,10,90230


Silver: Clean pit_stops

In [0]:
# Drop source and duration string (milliseconds column already has the value)
# merge_from_cdf: 1st run = full overwrite (creates table), subsequent runs = incremental MERGE
merge_from_cdf(
    "f1_bronze.pit_stops",
    "f1_silver.pit_stops",
    ["raceId", "driverId", "stop"],
    lambda df: df.drop("source", "duration", "_ingested_at")
)
print("f1_silver.pit_stops ready")
display(spark.table("f1_silver.pit_stops").limit(10))

  Merged 25712 rows into f1_silver.pit_stops  (Delta v11 -> v13)
f1_silver.pit_stops ready


raceId,driverId,stop,lap,time,milliseconds
841,1,1,16,17:28:24,23227.0
841,1,2,36,17:59:29,23199.0
841,2,1,15,17:27:41,22994.0
841,2,2,30,17:51:32,25098.0
841,3,1,16,17:29:00,23716.0
841,4,1,12,17:22:34,23251.0
841,4,2,27,17:46:04,24733.0
841,4,3,42,18:09:08,24181.0
841,5,1,17,17:31:11,24865.0
841,10,1,18,17:33:02,23792.0


Silver: Clean qualifying

In [0]:
from pyspark.sql.functions import split, col

# Drop source, convert q1/q2/q3 time strings ("M:SS.mmm") to integer milliseconds
# Nulls are preserved — they mean the driver was eliminated in an earlier session
# merge_from_cdf: 1st run = full overwrite (creates table), subsequent runs = incremental MERGE
merge_from_cdf(
    "f1_bronze.qualifying",
    "f1_silver.qualifying",
    ["qualifyId"],
    lambda df: df.drop("source", "_ingested_at")
        .withColumn("q1_ms", (split(col("q1"), ":")[0].cast("double") * 60000 + split(col("q1"), ":")[1].cast("double") * 1000).cast("int"))
        .withColumn("q2_ms", (split(col("q2"), ":")[0].cast("double") * 60000 + split(col("q2"), ":")[1].cast("double") * 1000).cast("int"))
        .withColumn("q3_ms", (split(col("q3"), ":")[0].cast("double") * 60000 + split(col("q3"), ":")[1].cast("double") * 1000).cast("int"))
        .drop("q1", "q2", "q3")
)
print("f1_silver.qualifying ready")
display(spark.table("f1_silver.qualifying").limit(10))

  Merged 22640 rows into f1_silver.qualifying  (Delta v10 -> v13)
f1_silver.qualifying ready


qualifyId,raceId,driverId,constructorId,number,position,q1_ms,q2_ms,q3_ms
1,18,1,1,22,1,86572,85187,86714
2,18,9,2,4,2,86103,85315,86869
3,18,5,1,23,3,85664,85452,87079
4,18,13,6,2,4,85994,85691,87178
5,18,2,2,3,5,85960,85518,87236
6,18,15,7,11,6,86427,86101,88527
7,18,3,3,7,7,86295,86059,88687
8,18,14,9,9,8,86381,86063,89041
9,18,10,7,12,9,86919,86164,89593
10,18,20,5,15,10,86702,85842,null


Silver: Clean races

In [0]:
from pyspark.sql.functions import col, to_date

# Drop url/source, convert all date columns to DATE type
# merge_from_cdf: 1st run = full overwrite (creates table), subsequent runs = incremental MERGE
merge_from_cdf(
    "f1_bronze.races",
    "f1_silver.races",
    ["raceId"],
    lambda df: df.drop("url", "source", "_ingested_at")
        .withColumn("date", to_date(col("date"), "yyyy-MM-dd"))
        .withColumn("fp1_date", to_date(col("fp1_date"), "yyyy-MM-dd"))
        .withColumn("fp2_date", to_date(col("fp2_date"), "yyyy-MM-dd"))
        .withColumn("fp3_date", to_date(col("fp3_date"), "yyyy-MM-dd"))
        .withColumn("quali_date", to_date(col("quali_date"), "yyyy-MM-dd"))
        .withColumn("sprint_date", to_date(col("sprint_date"), "yyyy-MM-dd"))
)
print("f1_silver.races ready")
display(spark.table("f1_silver.races").limit(10))

  Merged 2344 rows into f1_silver.races  (Delta v10 -> v12)
f1_silver.races ready


raceId,year,round,circuitId,name,date,time,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time
1,2009,1,1,Australian Grand Prix,2009-03-29,06:00:00,null,null,null,null,null,null,null,null,null,null
2,2009,2,2,Malaysian Grand Prix,2009-04-05,09:00:00,null,null,null,null,null,null,null,null,null,null
3,2009,3,17,Chinese Grand Prix,2009-04-19,07:00:00,null,null,null,null,null,null,null,null,null,null
4,2009,4,3,Bahrain Grand Prix,2009-04-26,12:00:00,null,null,null,null,null,null,null,null,null,null
5,2009,5,4,Spanish Grand Prix,2009-05-10,12:00:00,null,null,null,null,null,null,null,null,null,null
6,2009,6,6,Monaco Grand Prix,2009-05-24,12:00:00,null,null,null,null,null,null,null,null,null,null
7,2009,7,5,Turkish Grand Prix,2009-06-07,12:00:00,null,null,null,null,null,null,null,null,null,null
8,2009,8,9,British Grand Prix,2009-06-21,12:00:00,null,null,null,null,null,null,null,null,null,null
9,2009,9,20,German Grand Prix,2009-07-12,12:00:00,null,null,null,null,null,null,null,null,null,null
10,2009,10,11,Hungarian Grand Prix,2009-07-26,12:00:00,null,null,null,null,null,null,null,null,null,null


Silver: Clean results

In [0]:
from pyspark.sql.functions import col, when

# Drop source, decode positionText to finish_status enum, cast types
# merge_from_cdf: 1st run = full overwrite (creates table), subsequent runs = incremental MERGE
merge_from_cdf(
    "f1_bronze.results",
    "f1_silver.results",
    ["resultId"],
    lambda df: df.drop("source", "_ingested_at")
        .withColumn("position", col("position").cast("int"))
        .withColumn("number", col("number").cast("int"))
        .withColumn("finish_status",
            when(col("positionText") == "R", "Retired")
            .when(col("positionText") == "F", "Failed to qualify")
            .when(col("positionText") == "W", "Withdrew")
            .when(col("positionText") == "N", "Not classified")
            .when(col("positionText") == "D", "Disqualified")
            .when(col("positionText") == "E", "Excluded")
            .otherwise("Finished"))
        .drop("positionText")
)
print("f1_silver.results ready")
display(spark.table("f1_silver.results").limit(10))

  Merged 55180 rows into f1_silver.results  (Delta v11 -> v13)
f1_silver.results ready


resultId,raceId,driverId,constructorId,number,grid,position,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId,finish_status
1,18,1,1,22,1,1,1,10.0,58,1:34:50.616,5690616.0,39.0,2.0,1:27.452,218.3,1,Finished
2,18,2,2,3,5,2,2,8.0,58,+5.478,5696094.0,41.0,3.0,1:27.739,217.586,1,Finished
3,18,3,3,7,7,3,3,6.0,58,+8.163,5698779.0,41.0,5.0,1:28.090,216.719,1,Finished
4,18,4,4,5,11,4,4,5.0,58,+17.181,5707797.0,58.0,7.0,1:28.603,215.464,1,Finished
5,18,5,1,23,3,5,5,4.0,58,+18.014,5708630.0,43.0,1.0,1:27.418,218.385,1,Finished
6,18,6,3,8,13,6,6,3.0,57,null,null,50.0,14.0,1:29.639,212.974,11,Finished
7,18,7,5,14,17,7,7,2.0,55,null,null,54.0,8.0,1:29.534,213.224,5,Finished
8,18,8,6,1,15,8,8,1.0,53,null,null,20.0,4.0,1:27.903,217.18,5,Finished
9,18,9,2,4,2,null,9,0.0,47,null,null,15.0,9.0,1:28.753,215.1,4,Retired
10,18,10,7,12,18,null,10,0.0,43,null,null,23.0,13.0,1:29.558,213.166,3,Retired


Silver: Clean seasons

In [0]:
# Drop url and source columns
# merge_from_cdf: 1st run = full overwrite (creates table), subsequent runs = incremental MERGE
merge_from_cdf(
    "f1_bronze.seasons",
    "f1_silver.seasons",
    ["year"],
    lambda df: df.drop("url", "source", "_ingested_at")
)
print("f1_silver.seasons ready")
display(spark.table("f1_silver.seasons"))

  Merged 154 rows into f1_silver.seasons  (Delta v10 -> v12)
f1_silver.seasons ready


year
1950
1951
1952
1953
1954
1955
1956
1957
1958
1959


Silver: Clean sprint_results

In [0]:
from pyspark.sql.functions import col, when

# Drop source, decode positionText to finish_status, cast types
# merge_from_cdf: 1st run = full overwrite (creates table), subsequent runs = incremental MERGE
merge_from_cdf(
    "f1_bronze.sprint_results",
    "f1_silver.sprint_results",
    ["sprintResultId"],
    lambda df: df.drop("source", "_ingested_at")
        .withColumn("position", col("position").cast("int"))
        .withColumn("finish_status",
            when(col("positionText") == "R", "Retired")
            .when(col("positionText") == "F", "Failed to qualify")
            .when(col("positionText") == "W", "Withdrew")
            .when(col("positionText") == "N", "Not classified")
            .when(col("positionText") == "D", "Disqualified")
            .when(col("positionText") == "E", "Excluded")
            .otherwise("Finished"))
        .drop("positionText")
)
print("f1_silver.sprint_results ready")
display(spark.table("f1_silver.sprint_results").limit(10))

  Merged 1180 rows into f1_silver.sprint_results  (Delta v10 -> v12)
f1_silver.sprint_results ready


sprintResultId,raceId,driverId,constructorId,number,grid,position,positionOrder,points,laps,time,milliseconds,fastestLap,fastestLapTime,statusId,finish_status
1,1061,830,9,33,2,1,1,3,17,25:38.426,1538426.0,14.0,1:30.013,1,Finished
2,1061,1,131,44,1,2,2,2,17,+1.430,1539856.0,17.0,1:29.937,1,Finished
3,1061,822,131,77,3,3,3,1,17,+7.502,1545928.0,17.0,1:29.958,1,Finished
4,1061,844,6,16,4,4,4,0,17,+11.278,1549704.0,16.0,1:30.163,1,Finished
5,1061,846,1,4,6,5,5,0,17,+24.111,1562537.0,16.0,1:30.566,1,Finished
6,1061,817,1,3,7,6,6,0,17,+30.959,1569385.0,17.0,1:30.640,1,Finished
7,1061,4,214,14,11,7,7,0,17,+43.527,1581953.0,17.0,1:31.773,1,Finished
8,1061,20,117,5,10,8,8,0,17,+44.439,1582865.0,17.0,1:31.687,1,Finished
9,1061,847,3,63,8,9,9,0,17,+46.652,1585078.0,17.0,1:32.208,1,Finished
10,1061,839,214,31,13,10,10,0,17,+47.395,1585821.0,16.0,1:32.183,1,Finished


Silver: Clean status

In [0]:
from pyspark.sql.functions import col, when

# Drop source, add status_category to group 141 statuses into broader buckets
# merge_from_cdf: 1st run = full overwrite (creates table), subsequent runs = incremental MERGE
merge_from_cdf(
    "f1_bronze.status",
    "f1_silver.status",
    ["statusId"],
    lambda df: df.drop("source", "_ingested_at")
        .withColumn("status_category",
            when(col("status") == "Finished", "Finished")
            .when(col("status").like("+%"), "Finished")  # lapped but still running
            .when(col("status").isin("Accident", "Collision", "Spun off", "Collision damage"), "Incident")
            .when(col("status").isin("Did not qualify", "Did not prequalify", "Did not start"), "Did not participate")
            .when(col("status").isin("Withdrew", "Retired"), "Withdrawn/Retired")
            .when(col("status").isin("Disqualified", "Excluded"), "Disqualified/Excluded")
            .when(col("status") == "Fatal accident", "Fatal")
            .when(col("status").isin("107% Rule", "Safety", "Safety concerns"), "Regulation")
            .when(col("status") == "Not classified", "Not classified")
            .otherwise("Mechanical"))
)
print("f1_silver.status ready")
display(spark.table("f1_silver.status").limit(20))

  Merged 282 rows into f1_silver.status  (Delta v10 -> v12)
f1_silver.status ready


statusId,status,status_category
1,Finished,Finished
2,Disqualified,Disqualified/Excluded
3,Accident,Incident
4,Collision,Incident
5,Engine,Mechanical
6,Gearbox,Mechanical
7,Transmission,Mechanical
8,Clutch,Mechanical
9,Hydraulics,Mechanical
10,Electrical,Mechanical


Gold: Create schema & dimension tables

In [0]:
from pyspark.sql.functions import col

# Create the gold schema for Power BI-ready star schema
spark.sql("CREATE SCHEMA IF NOT EXISTS f1_gold")

# ── Dimension tables (one row per business entity, PK + descriptive columns) ──

# dim_circuits
spark.table("f1_silver.circuits").write.format("delta").mode("overwrite").saveAsTable("f1_gold.dim_circuits")
print("Saved f1_gold.dim_circuits")

# dim_drivers
spark.table("f1_silver.drivers").write.format("delta").mode("overwrite").saveAsTable("f1_gold.dim_drivers")
print("Saved f1_gold.dim_drivers")

# dim_constructors
spark.table("f1_silver.constructors").write.format("delta").mode("overwrite").saveAsTable("f1_gold.dim_constructors")
print("Saved f1_gold.dim_constructors")

# dim_races — join circuit name/country so Power BI can filter by circuit without a relationship
spark.table("f1_silver.races") \
    .join(spark.table("f1_silver.circuits").select("circuitId", col("name").alias("circuit_name"), "country", "location"), "circuitId", "left") \
    .write.format("delta").mode("overwrite").saveAsTable("f1_gold.dim_races")
print("Saved f1_gold.dim_races")

# dim_status
spark.table("f1_silver.status").write.format("delta").mode("overwrite").saveAsTable("f1_gold.dim_status")
print("Saved f1_gold.dim_status")

# dim_seasons
spark.table("f1_silver.seasons").write.format("delta").mode("overwrite").saveAsTable("f1_gold.dim_seasons")
print("Saved f1_gold.dim_seasons")

Saved f1_gold.dim_circuits
Saved f1_gold.dim_drivers
Saved f1_gold.dim_constructors
Saved f1_gold.dim_races
Saved f1_gold.dim_status
Saved f1_gold.dim_seasons


Gold: fact_race_results

In [0]:
from pyspark.sql.functions import col

# Denormalized fact table for Power BI 
# Joins results with races, drivers, constructors, circuits, and status so Power BI can slice by driver name, constructor, circuit country, season, etc. without building separate relationships. Foreign keys are kept for star-schema use.

results = spark.table("f1_silver.results")
races   = spark.table("f1_silver.races")
drivers = spark.table("f1_silver.drivers")
cons    = spark.table("f1_silver.constructors")
circ    = spark.table("f1_silver.circuits")
status  = spark.table("f1_silver.status")

fact_race_results = (
    results
    .join(races.select(
          col("raceId"), col("year").alias("season"), col("round"), col("name").alias("race_name"),
          col("date").alias("race_date"), col("circuitId")),
          "raceId", "left")
    .join(drivers.select(
          col("driverId"), col("full_name"), col("code"), col("nationality").alias("driver_nationality"),
          col("dob")),
          "driverId", "left")
    .join(cons.select(
          col("constructorId"), col("name").alias("constructor_name"), col("nationality").alias("constructor_nationality")),
          "constructorId", "left")
    .join(circ.select(
          col("circuitId"), col("name").alias("circuit_name"), col("country").alias("circuit_country")),
          "circuitId", "left")
    .join(status.select("statusId", col("status").alias("finish_detail"), "status_category"),
          "statusId", "left")
    .drop("statusId", "circuitId")
)

fact_race_results.write.format("delta").mode("overwrite").saveAsTable("f1_gold.fact_race_results")
print(f"Saved f1_gold.fact_race_results: {fact_race_results.count()} rows, {len(fact_race_results.columns)} columns")
display(spark.table("f1_gold.fact_race_results").limit(10))

Saved f1_gold.fact_race_results: 27590 rows, 31 columns


constructorId,driverId,raceId,resultId,number,grid,position,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,finish_status,season,round,race_name,race_date,full_name,code,driver_nationality,dob,constructor_name,constructor_nationality,circuit_name,circuit_country,finish_detail,status_category
1,1,18,1,22,1,1,1,10.0,58,1:34:50.616,5690616.0,39.0,2.0,1:27.452,218.3,Finished,2008,1,Australian Grand Prix,2008-03-16,Lewis Hamilton,HAM,British,1985-01-07,McLaren,British,Albert Park Grand Prix Circuit,Australia,Finished,Finished
2,2,18,2,3,5,2,2,8.0,58,+5.478,5696094.0,41.0,3.0,1:27.739,217.586,Finished,2008,1,Australian Grand Prix,2008-03-16,Nick Heidfeld,HEI,German,1977-05-10,BMW Sauber,German,Albert Park Grand Prix Circuit,Australia,Finished,Finished
3,3,18,3,7,7,3,3,6.0,58,+8.163,5698779.0,41.0,5.0,1:28.090,216.719,Finished,2008,1,Australian Grand Prix,2008-03-16,Nico Rosberg,ROS,German,1985-06-27,Williams,British,Albert Park Grand Prix Circuit,Australia,Finished,Finished
4,4,18,4,5,11,4,4,5.0,58,+17.181,5707797.0,58.0,7.0,1:28.603,215.464,Finished,2008,1,Australian Grand Prix,2008-03-16,Fernando Alonso,ALO,Spanish,1981-07-29,Renault,French,Albert Park Grand Prix Circuit,Australia,Finished,Finished
1,5,18,5,23,3,5,5,4.0,58,+18.014,5708630.0,43.0,1.0,1:27.418,218.385,Finished,2008,1,Australian Grand Prix,2008-03-16,Heikki Kovalainen,KOV,Finnish,1981-10-19,McLaren,British,Albert Park Grand Prix Circuit,Australia,Finished,Finished
3,6,18,6,8,13,6,6,3.0,57,null,null,50.0,14.0,1:29.639,212.974,Finished,2008,1,Australian Grand Prix,2008-03-16,Kazuki Nakajima,NAK,Japanese,1985-01-11,Williams,British,Albert Park Grand Prix Circuit,Australia,+1 Lap,Finished
5,7,18,7,14,17,7,7,2.0,55,null,null,54.0,8.0,1:29.534,213.224,Finished,2008,1,Australian Grand Prix,2008-03-16,Sébastien Bourdais,BOU,French,1979-02-28,Toro Rosso,Italian,Albert Park Grand Prix Circuit,Australia,Engine,Mechanical
6,8,18,8,1,15,8,8,1.0,53,null,null,20.0,4.0,1:27.903,217.18,Finished,2008,1,Australian Grand Prix,2008-03-16,Kimi Räikkönen,RAI,Finnish,1979-10-17,Ferrari,Italian,Albert Park Grand Prix Circuit,Australia,Engine,Mechanical
2,9,18,9,4,2,null,9,0.0,47,null,null,15.0,9.0,1:28.753,215.1,Retired,2008,1,Australian Grand Prix,2008-03-16,Robert Kubica,KUB,Polish,1984-12-07,BMW Sauber,German,Albert Park Grand Prix Circuit,Australia,Collision,Incident
7,10,18,10,12,18,null,10,0.0,43,null,null,23.0,13.0,1:29.558,213.166,Retired,2008,1,Australian Grand Prix,2008-03-16,Timo Glock,GLO,German,1982-03-18,Toyota,Japanese,Albert Park Grand Prix Circuit,Australia,Accident,Incident


Gold: fact_sprint_results

In [0]:
from pyspark.sql.functions import col

# Denormalized sprint fact table (same pattern as race results)

sprint = spark.table("f1_silver.sprint_results")

fact_sprint_results = (
    sprint
    .join(races.select(
          col("raceId"), col("year").alias("season"), col("round"), col("name").alias("race_name"),
          col("date").alias("race_date"), col("circuitId")),
          "raceId", "left")
    .join(drivers.select(
          col("driverId"), col("full_name"), col("code"), col("nationality").alias("driver_nationality")),
          "driverId", "left")
    .join(cons.select(
          col("constructorId"), col("name").alias("constructor_name"), col("nationality").alias("constructor_nationality")),
          "constructorId", "left")
    .join(circ.select(
          col("circuitId"), col("name").alias("circuit_name"), col("country").alias("circuit_country")),
          "circuitId", "left")
    .join(status.select("statusId", col("status").alias("finish_detail"), "status_category"),
          "statusId", "left")
    .drop("statusId", "circuitId")
)

fact_sprint_results.write.format("delta").mode("overwrite").saveAsTable("f1_gold.fact_sprint_results")
print(f"Saved f1_gold.fact_sprint_results: {fact_sprint_results.count()} rows, {len(fact_sprint_results.columns)} columns")
display(spark.table("f1_gold.fact_sprint_results").limit(10))

Saved f1_gold.fact_sprint_results: 590 rows, 28 columns


constructorId,driverId,raceId,sprintResultId,number,grid,position,positionOrder,points,laps,time,milliseconds,fastestLap,fastestLapTime,finish_status,season,round,race_name,race_date,full_name,code,driver_nationality,constructor_name,constructor_nationality,circuit_name,circuit_country,finish_detail,status_category
9,830,1061,1,33,2,1,1,3,17,25:38.426,1538426.0,14.0,1:30.013,Finished,2021,10,British Grand Prix,2021-07-18,Max Verstappen,VER,Dutch,Red Bull,Austrian,Silverstone Circuit,UK,Finished,Finished
131,1,1061,2,44,1,2,2,2,17,+1.430,1539856.0,17.0,1:29.937,Finished,2021,10,British Grand Prix,2021-07-18,Lewis Hamilton,HAM,British,Mercedes,German,Silverstone Circuit,UK,Finished,Finished
131,822,1061,3,77,3,3,3,1,17,+7.502,1545928.0,17.0,1:29.958,Finished,2021,10,British Grand Prix,2021-07-18,Valtteri Bottas,BOT,Finnish,Mercedes,German,Silverstone Circuit,UK,Finished,Finished
6,844,1061,4,16,4,4,4,0,17,+11.278,1549704.0,16.0,1:30.163,Finished,2021,10,British Grand Prix,2021-07-18,Charles Leclerc,LEC,Monegasque,Ferrari,Italian,Silverstone Circuit,UK,Finished,Finished
1,846,1061,5,4,6,5,5,0,17,+24.111,1562537.0,16.0,1:30.566,Finished,2021,10,British Grand Prix,2021-07-18,Lando Norris,NOR,British,McLaren,British,Silverstone Circuit,UK,Finished,Finished
1,817,1061,6,3,7,6,6,0,17,+30.959,1569385.0,17.0,1:30.640,Finished,2021,10,British Grand Prix,2021-07-18,Daniel Ricciardo,RIC,Australian,McLaren,British,Silverstone Circuit,UK,Finished,Finished
214,4,1061,7,14,11,7,7,0,17,+43.527,1581953.0,17.0,1:31.773,Finished,2021,10,British Grand Prix,2021-07-18,Fernando Alonso,ALO,Spanish,Alpine F1 Team,French,Silverstone Circuit,UK,Finished,Finished
117,20,1061,8,5,10,8,8,0,17,+44.439,1582865.0,17.0,1:31.687,Finished,2021,10,British Grand Prix,2021-07-18,Sebastian Vettel,VET,German,Aston Martin,British,Silverstone Circuit,UK,Finished,Finished
3,847,1061,9,63,8,9,9,0,17,+46.652,1585078.0,17.0,1:32.208,Finished,2021,10,British Grand Prix,2021-07-18,George Russell,RUS,British,Williams,British,Silverstone Circuit,UK,Finished,Finished
214,839,1061,10,31,13,10,10,0,17,+47.395,1585821.0,16.0,1:32.183,Finished,2021,10,British Grand Prix,2021-07-18,Esteban Ocon,OCO,French,Alpine F1 Team,French,Silverstone Circuit,UK,Finished,Finished


Gold: fact_driver_standings

In [0]:
from pyspark.sql.functions import col

# Enriched with driver name, race context

dst = spark.table("f1_silver.driver_standings")

fact_driver_standings = (
    dst
    .join(races.select(
          col("raceId"), col("year").alias("season"), col("round"), col("name").alias("race_name"),
          col("date").alias("race_date")),
          "raceId", "left")
    .join(drivers.select(
          col("driverId"), col("full_name"), col("code"), col("nationality").alias("driver_nationality")),
          "driverId", "left")
)

fact_driver_standings.write.format("delta").mode("overwrite").saveAsTable("f1_gold.fact_driver_standings")
print(f"Saved f1_gold.fact_driver_standings: {fact_driver_standings.count()} rows, {len(fact_driver_standings.columns)} columns")
display(spark.table("f1_gold.fact_driver_standings").limit(10))

Saved f1_gold.fact_driver_standings: 35718 rows, 13 columns


driverId,raceId,driverStandingsId,points,position,wins,season,round,race_name,race_date,full_name,code,driver_nationality
1,18,1,10.0,1.0,1,2008,1,Australian Grand Prix,2008-03-16,Lewis Hamilton,HAM,British
2,18,2,8.0,2.0,0,2008,1,Australian Grand Prix,2008-03-16,Nick Heidfeld,HEI,German
3,18,3,6.0,3.0,0,2008,1,Australian Grand Prix,2008-03-16,Nico Rosberg,ROS,German
4,18,4,5.0,4.0,0,2008,1,Australian Grand Prix,2008-03-16,Fernando Alonso,ALO,Spanish
5,18,5,4.0,5.0,0,2008,1,Australian Grand Prix,2008-03-16,Heikki Kovalainen,KOV,Finnish
6,18,6,3.0,6.0,0,2008,1,Australian Grand Prix,2008-03-16,Kazuki Nakajima,NAK,Japanese
7,18,7,2.0,7.0,0,2008,1,Australian Grand Prix,2008-03-16,Sébastien Bourdais,BOU,French
8,18,8,1.0,8.0,0,2008,1,Australian Grand Prix,2008-03-16,Kimi Räikkönen,RAI,Finnish
1,19,9,14.0,1.0,1,2008,2,Malaysian Grand Prix,2008-03-23,Lewis Hamilton,HAM,British
2,19,10,11.0,3.0,0,2008,2,Malaysian Grand Prix,2008-03-23,Nick Heidfeld,HEI,German


Gold: fact_constructor_standings

In [0]:
from pyspark.sql.functions import col

# Enriched with constructor name, race context

cst = spark.table("f1_silver.constructor_standings")

fact_constructor_standings = (
    cst
    .join(races.select(
          col("raceId"), col("year").alias("season"), col("round"), col("name").alias("race_name"),
          col("date").alias("race_date")),
          "raceId", "left")
    .join(cons.select(
          col("constructorId"), col("name").alias("constructor_name"), col("nationality").alias("constructor_nationality")),
          "constructorId", "left")
)

fact_constructor_standings.write.format("delta").mode("overwrite").saveAsTable("f1_gold.fact_constructor_standings")
print(f"Saved f1_gold.fact_constructor_standings: {fact_constructor_standings.count()} rows, {len(fact_constructor_standings.columns)} columns")
display(spark.table("f1_gold.fact_constructor_standings").limit(10))

Saved f1_gold.fact_constructor_standings: 13807 rows, 12 columns


constructorId,raceId,constructorStandingsId,points,position,wins,season,round,race_name,race_date,constructor_name,constructor_nationality
1,18,1,14.0,1,1,2008,1,Australian Grand Prix,2008-03-16,McLaren,British
2,18,2,8.0,3,0,2008,1,Australian Grand Prix,2008-03-16,BMW Sauber,German
3,18,3,9.0,2,0,2008,1,Australian Grand Prix,2008-03-16,Williams,British
4,18,4,5.0,4,0,2008,1,Australian Grand Prix,2008-03-16,Renault,French
5,18,5,2.0,5,0,2008,1,Australian Grand Prix,2008-03-16,Toro Rosso,Italian
6,18,6,1.0,6,0,2008,1,Australian Grand Prix,2008-03-16,Ferrari,Italian
1,19,7,24.0,1,1,2008,2,Malaysian Grand Prix,2008-03-23,McLaren,British
2,19,8,19.0,2,0,2008,2,Malaysian Grand Prix,2008-03-23,BMW Sauber,German
3,19,9,9.0,4,0,2008,2,Malaysian Grand Prix,2008-03-23,Williams,British
4,19,10,6.0,5,0,2008,2,Malaysian Grand Prix,2008-03-23,Renault,French


Gold: fact_qualifying

In [0]:
from pyspark.sql.functions import col

# Enriched with race, driver, constructor context

qual = spark.table("f1_silver.qualifying")

fact_qualifying = (
    qual
    .join(races.select(
          col("raceId"), col("year").alias("season"), col("round"), col("name").alias("race_name"),
          col("date").alias("race_date"), col("circuitId")),
          "raceId", "left")
    .join(drivers.select(
          col("driverId"), col("full_name"), col("code"), col("nationality").alias("driver_nationality")),
          "driverId", "left")
    .join(cons.select(
          col("constructorId"), col("name").alias("constructor_name"), col("nationality").alias("constructor_nationality")),
          "constructorId", "left")
    .join(circ.select(
          col("circuitId"), col("name").alias("circuit_name"), col("country").alias("circuit_country")),
          "circuitId", "left")
    .drop("circuitId")
)

fact_qualifying.write.format("delta").mode("overwrite").saveAsTable("f1_gold.fact_qualifying")
print(f"Saved f1_gold.fact_qualifying: {fact_qualifying.count()} rows, {len(fact_qualifying.columns)} columns")
display(spark.table("f1_gold.fact_qualifying").limit(10))

Saved f1_gold.fact_qualifying: 11320 rows, 20 columns


constructorId,driverId,raceId,qualifyId,number,position,q1_ms,q2_ms,q3_ms,season,round,race_name,race_date,full_name,code,driver_nationality,constructor_name,constructor_nationality,circuit_name,circuit_country
1,1,18,1,22,1,86572,85187,86714,2008,1,Australian Grand Prix,2008-03-16,Lewis Hamilton,HAM,British,McLaren,British,Albert Park Grand Prix Circuit,Australia
2,9,18,2,4,2,86103,85315,86869,2008,1,Australian Grand Prix,2008-03-16,Robert Kubica,KUB,Polish,BMW Sauber,German,Albert Park Grand Prix Circuit,Australia
1,5,18,3,23,3,85664,85452,87079,2008,1,Australian Grand Prix,2008-03-16,Heikki Kovalainen,KOV,Finnish,McLaren,British,Albert Park Grand Prix Circuit,Australia
6,13,18,4,2,4,85994,85691,87178,2008,1,Australian Grand Prix,2008-03-16,Felipe Massa,MAS,Brazilian,Ferrari,Italian,Albert Park Grand Prix Circuit,Australia
2,2,18,5,3,5,85960,85518,87236,2008,1,Australian Grand Prix,2008-03-16,Nick Heidfeld,HEI,German,BMW Sauber,German,Albert Park Grand Prix Circuit,Australia
7,15,18,6,11,6,86427,86101,88527,2008,1,Australian Grand Prix,2008-03-16,Jarno Trulli,TRU,Italian,Toyota,Japanese,Albert Park Grand Prix Circuit,Australia
3,3,18,7,7,7,86295,86059,88687,2008,1,Australian Grand Prix,2008-03-16,Nico Rosberg,ROS,German,Williams,British,Albert Park Grand Prix Circuit,Australia
9,14,18,8,9,8,86381,86063,89041,2008,1,Australian Grand Prix,2008-03-16,David Coulthard,COU,British,Red Bull,Austrian,Albert Park Grand Prix Circuit,Australia
7,10,18,9,12,9,86919,86164,89593,2008,1,Australian Grand Prix,2008-03-16,Timo Glock,GLO,German,Toyota,Japanese,Albert Park Grand Prix Circuit,Australia
5,20,18,10,15,10,86702,85842,null,2008,1,Australian Grand Prix,2008-03-16,Sebastian Vettel,VET,German,Toro Rosso,Italian,Albert Park Grand Prix Circuit,Australia


Gold: fact_lap_times

In [0]:
from pyspark.sql.functions import col

# Enriched with race and driver context
# Keep it lean for Power BI.

laps = spark.table("f1_silver.lap_times")

fact_lap_times = (
    laps
    .join(races.select(
          col("raceId"), col("year").alias("season"), col("round"), col("name").alias("race_name"),
          col("date").alias("race_date"), col("circuitId")),
          "raceId", "left")
    .join(drivers.select(
          col("driverId"), col("full_name"), col("code")),
          "driverId", "left")
    .join(circ.select(
          col("circuitId"), col("name").alias("circuit_name"), col("country").alias("circuit_country")),
          "circuitId", "left")
    .drop("circuitId")
)

fact_lap_times.write.format("delta").mode("overwrite").saveAsTable("f1_gold.fact_lap_times")
print(f"Saved f1_gold.fact_lap_times: {fact_lap_times.count()} rows, {len(fact_lap_times.columns)} columns")
display(spark.table("f1_gold.fact_lap_times").limit(10))

Saved f1_gold.fact_lap_times: 634079 rows, 13 columns


driverId,raceId,lap,position,milliseconds,season,round,race_name,race_date,full_name,code,circuit_name,circuit_country
1,1,26,10,92198,2009,1,Australian Grand Prix,2009-03-29,Lewis Hamilton,HAM,Albert Park Grand Prix Circuit,Australia
2,1,45,15,94324,2009,1,Australian Grand Prix,2009-03-29,Nick Heidfeld,HEI,Albert Park Grand Prix Circuit,Australia
3,1,50,6,90277,2009,1,Australian Grand Prix,2009-03-29,Nico Rosberg,ROS,Albert Park Grand Prix Circuit,Australia
7,1,33,15,91551,2009,1,Australian Grand Prix,2009-03-29,Sébastien Bourdais,BOU,Albert Park Grand Prix Circuit,Australia
8,1,41,8,89996,2009,1,Australian Grand Prix,2009-03-29,Kimi Räikkönen,RAI,Albert Park Grand Prix Circuit,Australia
10,1,10,13,91589,2009,1,Australian Grand Prix,2009-03-29,Timo Glock,GLO,Albert Park Grand Prix Circuit,Australia
10,1,47,6,89983,2009,1,Australian Grand Prix,2009-03-29,Timo Glock,GLO,Albert Park Grand Prix Circuit,Australia
12,1,11,7,91748,2009,1,Australian Grand Prix,2009-03-29,Nelson Piquet Jr.,PIQ,Albert Park Grand Prix Circuit,Australia
17,1,48,16,92184,2009,1,Australian Grand Prix,2009-03-29,Mark Webber,WEB,Albert Park Grand Prix Circuit,Australia
21,1,42,10,90230,2009,1,Australian Grand Prix,2009-03-29,Giancarlo Fisichella,FIS,Albert Park Grand Prix Circuit,Australia


Gold: fact_pit_stops

In [0]:
from pyspark.sql.functions import col

# Enriched with race and driver context

pits = spark.table("f1_silver.pit_stops")

fact_pit_stops = (
    pits
    .join(races.select(
          col("raceId"), col("year").alias("season"), col("round"), col("name").alias("race_name"),
          col("date").alias("race_date"), col("circuitId")),
          "raceId", "left")
    .join(drivers.select(
          col("driverId"), col("full_name"), col("code")),
          "driverId", "left")
    .join(circ.select(
          col("circuitId"), col("name").alias("circuit_name"), col("country").alias("circuit_country")),
          "circuitId", "left")
    .drop("circuitId")
)

fact_pit_stops.write.format("delta").mode("overwrite").saveAsTable("f1_gold.fact_pit_stops")
print(f"Saved f1_gold.fact_pit_stops: {fact_pit_stops.count()} rows, {len(fact_pit_stops.columns)} columns")
display(spark.table("f1_gold.fact_pit_stops").limit(10))

Saved f1_gold.fact_pit_stops: 12856 rows, 14 columns


driverId,raceId,stop,lap,time,milliseconds,season,round,race_name,race_date,full_name,code,circuit_name,circuit_country
1,841,1,16,17:28:24,23227.0,2011,1,Australian Grand Prix,2011-03-27,Lewis Hamilton,HAM,Albert Park Grand Prix Circuit,Australia
1,841,2,36,17:59:29,23199.0,2011,1,Australian Grand Prix,2011-03-27,Lewis Hamilton,HAM,Albert Park Grand Prix Circuit,Australia
2,841,1,15,17:27:41,22994.0,2011,1,Australian Grand Prix,2011-03-27,Nick Heidfeld,HEI,Albert Park Grand Prix Circuit,Australia
2,841,2,30,17:51:32,25098.0,2011,1,Australian Grand Prix,2011-03-27,Nick Heidfeld,HEI,Albert Park Grand Prix Circuit,Australia
3,841,1,16,17:29:00,23716.0,2011,1,Australian Grand Prix,2011-03-27,Nico Rosberg,ROS,Albert Park Grand Prix Circuit,Australia
4,841,1,12,17:22:34,23251.0,2011,1,Australian Grand Prix,2011-03-27,Fernando Alonso,ALO,Albert Park Grand Prix Circuit,Australia
4,841,2,27,17:46:04,24733.0,2011,1,Australian Grand Prix,2011-03-27,Fernando Alonso,ALO,Albert Park Grand Prix Circuit,Australia
4,841,3,42,18:09:08,24181.0,2011,1,Australian Grand Prix,2011-03-27,Fernando Alonso,ALO,Albert Park Grand Prix Circuit,Australia
5,841,1,17,17:31:11,24865.0,2011,1,Australian Grand Prix,2011-03-27,Heikki Kovalainen,KOV,Albert Park Grand Prix Circuit,Australia
10,841,1,18,17:33:02,23792.0,2011,1,Australian Grand Prix,2011-03-27,Timo Glock,GLO,Albert Park Grand Prix Circuit,Australia


Gold: Verify all tables

In [0]:
# List all gold tables with row counts

gold_tables = spark.sql("SHOW TABLES IN f1_gold").filter("isTemporary = false").collect()

print(f"f1_gold schema: {len(gold_tables)} tables\n")
print(f"{'Table':<35} {'Rows':>10}  {'Cols':>5}")
print("-" * 55)
for row in sorted(gold_tables, key=lambda r: r.tableName):
    tbl = f"f1_gold.{row.tableName}"
    cnt = spark.table(tbl).count()
    ncol = len(spark.table(tbl).columns)
    print(f"{tbl:<35} {cnt:>10,}  {ncol:>5}")

f1_gold schema: 13 tables

Table                                     Rows   Cols
-------------------------------------------------------
f1_gold.dim_circuits                        78      8
f1_gold.dim_constructors                   214      4
f1_gold.dim_drivers                        881      9
f1_gold.dim_races                        1,172     20
f1_gold.dim_seasons                         77      1
f1_gold.dim_status                         141      3
f1_gold.fact_constructor_standings      13,807     12
f1_gold.fact_driver_standings           35,718     13
f1_gold.fact_lap_times                 634,079     13
f1_gold.fact_pit_stops                  12,856     14
f1_gold.fact_qualifying                 11,320     20
f1_gold.fact_race_results               27,590     31
f1_gold.fact_sprint_results                590     28
